In [ ]:
import torch

inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

In [ ]:
x_2=inputs[1]
d_in=inputs.shape[1]
d_out=2

In [ ]:
torch.manual_seed(123)
W_query=torch.nn.Parameter(torch.randn(d_in,d_out),requires_grad=False)
W_key=torch.nn.Parameter(torch.randn(d_in,d_out),requires_grad=False)
W_value=torch.nn.Parameter(torch.randn(d_in,d_out),requires_grad=False)

In [ ]:
print(W_query)

Parameter containing:
tensor([[-0.1115,  0.1204],
        [-0.3696, -0.2404],
        [-1.1969,  0.2093]])


In [ ]:
print(W_key)

Parameter containing:
tensor([[-0.9724, -0.7550],
        [ 0.3239, -0.1085],
        [ 0.2103, -0.3908]])


In [ ]:
print(W_value)

Parameter containing:
tensor([[ 0.2350,  0.6653],
        [ 0.3528,  0.9728],
        [-0.0386, -0.8861]])


In [ ]:
query_2=x_2@ W_query
key_2=x_2@ W_key
value_2=x_2@ W_value
print(query_2)
print(key_2)
print(value_2)

tensor([-1.1729, -0.0048])
tensor([-0.1142, -0.7676])
tensor([0.4107, 0.6274])


In [ ]:
keys=inputs@ W_key
values=inputs@ W_value
queries=inputs@ W_query
print(keys)

tensor([[-0.1823, -0.6888],
        [-0.1142, -0.7676],
        [-0.1443, -0.7728],
        [ 0.0434, -0.3580],
        [-0.6467, -0.6476],
        [ 0.3262, -0.3395]])


In [ ]:
print(values)

tensor([[ 0.1196, -0.3566],
        [ 0.4107,  0.6274],
        [ 0.4091,  0.6390],
        [ 0.2436,  0.4182],
        [ 0.2653,  0.6668],
        [ 0.2728,  0.3242]])


In [ ]:
print(queries)

tensor([[-1.1686,  0.2019],
        [-1.1729, -0.0048],
        [-1.1438, -0.0018],
        [-0.6339, -0.0439],
        [-0.2979,  0.0535],
        [-0.9596, -0.0712]])


In [ ]:
keys_2=keys[1]
atten_score_22= query_2.dot(keys_2)
print(atten_score_22)

tensor(0.1376)


In [ ]:
atten_scores_2=query_2 @ keys.T
print(atten_scores_2)

tensor([ 0.2172,  0.1376,  0.1730, -0.0491,  0.7616, -0.3809])


In [ ]:
atten_scores=queries @ keys.T
print(atten_scores)

tensor([[ 0.0740, -0.0216,  0.0126, -0.1230,  0.6250, -0.4498],
        [ 0.2172,  0.1376,  0.1730, -0.0491,  0.7616, -0.3809],
        [ 0.2098,  0.1320,  0.1665, -0.0489,  0.7408, -0.3725],
        [ 0.1458,  0.1061,  0.1254, -0.0118,  0.4384, -0.1919],
        [ 0.0175, -0.0071,  0.0017, -0.0321,  0.1580, -0.1153],
        [ 0.2240,  0.1642,  0.1935, -0.0161,  0.6667, -0.2888]])


In [ ]:
d_k=keys.shape[-1]
atten_weights_2=torch.softmax(atten_scores_2/d_k**0.5,dim=-1)
print(atten_weights_2)

tensor([0.1704, 0.1611, 0.1652, 0.1412, 0.2505, 0.1117])


In [ ]:
contex_vec_2=atten_weights_2@ values
print(contex_vec_2)

tensor([0.2854, 0.4081])


In [ ]:
import torch.nn as nn

class SelfAttention_v1(nn.Module):

    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key   = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        keys = x @ self.W_key
        queries = x @ self.W_query
        values = x @ self.W_value

        attn_scores = queries @ keys.T # omega
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )

        context_vec = attn_weights @ values
        return context_vec

In [ ]:
torch.manual_seed(123)
sa_v1=SelfAttention_v1(d_in,d_out)
print(sa_v1(inputs))

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


In [ ]:
class SelfAttention_v2(nn.Module):

    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

        context_vec = attn_weights @ values
        return context_vec

In [ ]:
torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


Casual Self Attention

In [ ]:
queries=sa_v2(inputs)
keys=sa_v2.W_key(inputs)
atten_scores=queries @ keys.T
atten_weights=torch.softmax(atten_scores/keys.shape[-1]**0.5,dim=1)
print(atten_weights)

tensor([[0.1636, 0.1662, 0.1662, 0.1683, 0.1682, 0.1676],
        [0.1635, 0.1662, 0.1662, 0.1683, 0.1681, 0.1676],
        [0.1635, 0.1662, 0.1662, 0.1683, 0.1681, 0.1676],
        [0.1635, 0.1662, 0.1663, 0.1683, 0.1680, 0.1677],
        [0.1635, 0.1662, 0.1663, 0.1683, 0.1680, 0.1677],
        [0.1635, 0.1662, 0.1662, 0.1683, 0.1680, 0.1677]],
       grad_fn=<SoftmaxBackward0>)


In [ ]:
context_length=atten_scores.shape[0]
mask_simple=torch.tril(torch.ones(context_length,context_length))
print(mask_simple)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


In [ ]:
masked_simple=atten_weights*mask_simple
print(masked_simple)

tensor([[0.1636, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1635, 0.1662, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1635, 0.1662, 0.1662, 0.0000, 0.0000, 0.0000],
        [0.1635, 0.1662, 0.1663, 0.1683, 0.0000, 0.0000],
        [0.1635, 0.1662, 0.1663, 0.1683, 0.1680, 0.0000],
        [0.1635, 0.1662, 0.1662, 0.1683, 0.1680, 0.1677]],
       grad_fn=<MulBackward0>)


In [ ]:
row_sums=masked_simple.sum(dim=1,keepdim=True)
masked_simple_norm=masked_simple/row_sums
print(masked_simple_norm)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4960, 0.5040, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3297, 0.3351, 0.3352, 0.0000, 0.0000, 0.0000],
        [0.2462, 0.2502, 0.2503, 0.2534, 0.0000, 0.0000],
        [0.1965, 0.1997, 0.1998, 0.2022, 0.2018, 0.0000],
        [0.1635, 0.1662, 0.1662, 0.1683, 0.1680, 0.1677]],
       grad_fn=<DivBackward0>)


Avoiding Data Leakage

In [ ]:
mask=torch.triu(torch.ones(context_length,context_length),diagonal=1)
masked= atten_scores.masked_fill(mask.bool(),-torch.inf)
print(masked)

tensor([[-0.0519,    -inf,    -inf,    -inf,    -inf,    -inf],
        [-0.0518, -0.0291,    -inf,    -inf,    -inf,    -inf],
        [-0.0518, -0.0291, -0.0287,    -inf,    -inf,    -inf],
        [-0.0514, -0.0283, -0.0279, -0.0106,    -inf,    -inf],
        [-0.0513, -0.0280, -0.0277, -0.0105, -0.0134,    -inf],
        [-0.0516, -0.0287, -0.0283, -0.0109, -0.0131, -0.0161]],
       grad_fn=<MaskedFillBackward0>)


In [ ]:
atten_weights=torch.softmax(masked/ keys.shape[-1]**0.5,dim=1)
print(atten_weights)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4960, 0.5040, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3297, 0.3351, 0.3352, 0.0000, 0.0000, 0.0000],
        [0.2462, 0.2502, 0.2503, 0.2534, 0.0000, 0.0000],
        [0.1965, 0.1997, 0.1998, 0.2022, 0.2018, 0.0000],
        [0.1635, 0.1662, 0.1662, 0.1683, 0.1680, 0.1677]],
       grad_fn=<SoftmaxBackward0>)


Implementing the dropout

In [ ]:
torch.manual_seed(123)
dropout=torch.nn.Dropout(0.5)
example=torch.ones(6,6)
print(dropout(example))

tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])


In [ ]:
 torch.manual_seed(123)
 print(dropout(atten_weights))

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.6595, 0.6702, 0.6704, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.5004, 0.5006, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.3995, 0.0000, 0.4044, 0.0000, 0.0000],
        [0.0000, 0.3324, 0.3325, 0.3366, 0.3361, 0.0000]],
       grad_fn=<MulBackward0>)


Implementing the Casual Attention Class

In [ ]:
batch=torch.stack((inputs,inputs))
print(batch.shape)


torch.Size([2, 6, 3])


In [ ]:
class CausalAttention(nn.Module):

    def __init__(self, d_in, d_out, context_length,
                 dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1)) # New

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2)
        attn_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        attn_weights = self.dropout(attn_weights)

        context_vec = attn_weights @ values
        return context_vec

In [ ]:
torch.manual_seed(123)
context_length=batch.shape[1]
ca=CausalAttention(d_in,d_out,context_length,0.0)
context_vecs=ca(batch)
print("context_vecs.shape",context_vecs.shape)


context_vecs.shape torch.Size([2, 6, 2])


In [ ]:
print(context_vecs)

tensor([[[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]],

        [[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]]], grad_fn=<UnsafeViewBackward0>)
